In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations
from scipy import stats

tidy = pd.read_csv("data/tidy.csv")

In [ ]:
from scipy import stats

def get(construct, variant, experiment, condition):
    m = ((tidy.construct == construct) & (tidy.variant == variant) &
         (tidy.experiment == experiment) & (tidy.condition == condition))
    return tidy.loc[m, "pct_change"]

comparisons = {
    "GFP WT vs K125R":   (get("GFP","WT","response","55"),   get("GFP","K125","response","55")),
    "cpGFP WT vs K125R": (get("cpGFP","WT","response","55"), get("cpGFP","K125","response","55")),
    "GFP WT vs LL":      (get("GFP","WT","response","55"),   get("GFP","LL","response","55")),
}

for name, (a, b) in comparisons.items():
    r = stats.mannwhitneyu(a, b, alternative="two-sided", method="exact")
    print(name, "U =", r.statistic, "p =", round(r.pvalue, 4))

In [ ]:
import numpy as np
from itertools import combinations

def dr_wide(construct):
    d = tidy[(tidy.construct == construct) & (tidy.experiment == "DR")]
    return d.pivot(index="cell_id", columns="condition", values="pct_change")

def friedman_dunn(wide):
    n, k = wide.shape
    f = stats.friedmanchisquare(*[wide[c] for c in wide.columns])
    mean_ranks = wide.rank(axis=1).mean()
    se = np.sqrt(k * (k + 1) / (6 * n))
    print(f"n = {n}, Friedman = {f.statistic:.3f}, p = {f.pvalue:.4f}")
    pairs = list(combinations(wide.columns, 2))
    for a, b in pairs:
        z = abs(mean_ranks[a] - mean_ranks[b]) / se
        p = min(2 * (1 - stats.norm.cdf(z)) * len(pairs),
        print(f"  {a} vs {b}: adj p = {p:.4f}")

friedman_dunn(dr_wide("GFP"))
friedman_dunn(dr_wide("cpGFP"))